# State-of-the-Art Survey: Hybrid Classical–Quantum Reinforcement Learning

**Purpose:** Team reference document mapping the current landscape of quantum RL, with a focus on hybrid approaches that are most relevant to the design of a quantum-enhanced Implicit Q-Learning algorithm (Q-IQL).

## Table of contents

1. Brief Overview
2. Background and the standard VQC template
3. Method families
   - 3.1 Value-based QRL (Q-function as VQC)
   - 3.2 Policy-gradient QRL
   - 3.3 Actor–critic QRL
   - 3.4 Offline QRL
   - 3.5 Fault-tolerant / provable-advantage QRL (brief)
4. Component-level quantization patterns (how V, Q, π, A have been quantized)
5. Design building blocks
   - 5.1 State encoding
   - 5.2 Ansatz choice
   - 5.3 Measurement & readout
6. Current limitations
7. Open problems and implications for Q-IQL
8. References

---

## 1. Brief Overview

Hybrid classical–quantum RL (QRL) in its NISQ-era form almost always follows the same template: a parametrized / variational quantum circuit (PQC / VQC) replaces one or more neural-network function approximators inside an otherwise classical RL algorithm, and is trained by classical gradient descent through the parameter-shift rule.

Five broad method families exist today:

1. **Value-based QRL** — a VQC approximates Q(s, a) or V(s) inside DQN-style loops. This is by far the most studied family (Chen et al. 2020; Lockwood & Si 2020; Skolik, Jerbi & Dunjko 2022).
2. **Policy-gradient QRL** — a VQC defines a stochastic policy π(a|s), either by reading the Born-rule distribution directly (RAW-PQC) or by feeding expectation values into a softmax (SOFTMAX-PQC), and is trained by REINFORCE / PPO variants (Jerbi et al. 2021).
3. **Actor–critic QRL** — the actor, the critic, or both are replaced by VQCs (Lan 2021 — VQ-SAC; Kölle et al. 2024 — QA2C; Wu et al. 2020/2025 — Q-DDPG for quantum-control targets).
4. **Offline QRL** — only one published algorithm to date, CQ2L (Cheng, Zhang, Shen & Tao, AAAI 2023), which is a quantum analogue of CQL. This is the closest prior work to Q-IQL.
5. **Fault-tolerant / provable-advantage QRL** — quantum policy iteration and quantum-accessible-environment speed-ups that require oracle access or error-corrected hardware (Cherrat, Kerenidis & Prakash 2023; Wang et al. 2021; Dunjko et al. 2017).

Three observations frame the rest of this document:

- **No published QRL algorithm uses an expectile-regression value head, advantage-weighted BC, or any other IQL-specific construction.**
- **Most QRL results live in small discrete-action toy domains** (CartPole, FrozenLake, MountainCar, Acrobot, grid-worlds, Blackjack). D4RL-scale continuous-control offline RL has not been demonstrated with VQCs. This is both a risk and an opportunity for the project.
- **Trainability pathologies are the dominant practical blocker**: barren plateaus, noise-induced flattening of the loss landscape, and the specific Q-learning instabilities documented by Franz et al. (2023).

---

## 2. Background and the standard VQC template

### 2.1 Why hybrid

In the NISQ era — limited qubit counts, short coherence times, and significant gate/readout noise — the dominant paradigm is to use a PQC only for the step of the computation where one expects the model class to buy expressivity, and to leave everything else classical. In RL this typically means: keep the replay buffer, optimizer, target-network mechanics, environment loop and loss bookkeeping classical; use the VQC only to evaluate a function on a state (or state–action) input.

### 2.2 The standard VQC template

<img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTmqovzYY1jv9f5KRTcaq0qAmcqb58B9o-AUA&s" width="600">

Nearly every hybrid QRL paper composes three blocks in sequence:

1. **State encoding** `U_enc(s)`. Maps a classical state vector s ∈ ℝⁿ into a quantum state on a small register (typically 2–8 qubits). Three encoding families dominate:
   - *Angle / Pauli-rotation encoding*: each component of s becomes the angle of a single-qubit rotation. Cheap, shallow, but each feature only appears with frequency 1 in the function's Fourier expansion.
   - *Data re-uploading* (Pérez-Salinas et al. 2020): the encoding gates are interleaved with trainable layers and repeated L times. Schuld, Sweke & Meyer (2021) showed that repeating the encoding yields a partial Fourier series whose accessible frequency spectrum grows with L, so data re-uploading is the standard way to buy expressivity on few qubits.


<img src="https://i.imgur.com/wWsXeEe.png" width="600">
<br><br>
<img src="https://i.imgur.com/1970Z7F.png" width="600">


2. **Variational ansatz** `U_var(θ)`. A fixed architecture of parametrized single-qubit rotations plus entangling gates (CNOT or CZ rings, hardware-efficient ansätze, equivariant circuits, etc.). The parameters θ are trained classically via parameter-shift gradients.

3. **Measurement / readout.** The quantum output is a set of expectation values ⟨O_i⟩ of observables O_i (usually Pauli strings). For value-based methods there is typically one observable per action (discrete) or per output dimension (continuous). A scalar scaling / offset is almost always added on top, trained jointly.

General Hybrid QRL template



<img src="https://i.imgur.com/RU8v2u6.png" width="650">



---

## 3. Method families

### 3.1 Value-based QRL

#### Chen et al. 2020 — *Variational quantum circuits for deep reinforcement learning*

- First widely cited VQC-based deep-Q. Replaces the DQN MLP by a VQC keeping all other DQN machinery (experience replay, target network, ε-greedy) classical.
- Benchmarks: FrozenLake and a cognitive-radio environment. Demonstrates that the VQC can learn discrete-action policies with far fewer trainable parameters than the MLP it replaces.
- Contribution: establishes the "swap MLP for VQC" pattern that every later paper uses.

#### Lockwood & Si 2020 — *Reinforcement Learning with Quantum Variational Circuits*

- Extends Chen et al.'s approach to both pure-quantum and hybrid variants on *continuous-state* environments (CartPole and Blackjack).
- Contributes two encoding schemes — *Scaled encoding* and *Directional encoding* — chosen to handle unbounded and bounded state components respectively. On Blackjack the Scaled scheme preserves magnitude information that is policy-relevant (20 points ≠ 1 point); on CartPole the Directional scheme handles the unbounded state range.


<img src="https://i.imgur.com/roqbM3e.png" width="400">


- Reports that both pure-VQC and hybrid-VQC DQN/Double-DQN agents solve the tasks with smaller parameter counts than neural baselines. The pure-quantum CartPole agent slightly beats the hybrid one, hypothesised as the hybrid converging to a faster but local optimum.
- Takeaway for Q-IQL: *encoding choice has a policy-level effect, not just a representational one.* Magnitude-preserving encodings are necessary when the value function is sensitive to the scale of a feature.

#### Skolik, Jerbi & Dunjko 2022 — *Quantum agents in the Gym*

- de-facto reference implementation of VQC-DQN in the community (used by TensorFlow Quantum tutorials and PennyLane demos).
- Systematic ablations on data encoding and readout strategies (trainable input and output scaling factors, different observables). The central empirical claim: for a VQC-based Q-learner, *the choice of observables matters more than the number of parameters*. Observables should be matched to the structure of the target Q-values — for example, local Pauli-Z observables on the action-encoding qubits, multiplied by trainable output weights that rescale the bounded ⟨Z⟩ ∈ [−1, 1] range to the (unbounded) Q-value range.
- Performs an extensive NN-vs-VQC hyperparameter search on CartPole at matched parameter counts and reports that architectural choices (encoding + observable + entangling pattern) dominate the count. Also gives a formal separation argument extending earlier Jerbi et al. policy-gradient separations to Q-learning on restricted classes of environments.
- Introduces the data re-uploading PQC as a Q-function approximator — this is the specific architecture we should probably start from for Q(s, a) in Q-IQL.

#### Franz et al. 2023 — *Uncovering instabilities in variational-quantum deep Q-networks*

- The paper Q-IQL implementers most need to read. Runs large-scale re-implementations of VQ-DQN and documents two failure modes:
  - **Unbounded drift of VQC parameters.** Because the loss landscape is periodic in each parameter (period 2π for single-qubit rotations), there are infinitely many equivalent minima. Under standard Adam-style optimisers with no regularisation, the parameters drift monotonically to very large values instead of settling in one local copy, and this drift correlates with policy divergence.
  - **Reproducibility of prior results is fragile.** Many reported VQ-DQN successes in the literature do not reproduce cleanly across seeds and framework versions.
- They also run on real IBM hardware and find that shot noise and device noise amplify the instabilities.
- Explicit lesson for Q-IQL: assume from the start that value-based quantum training is *less stable than its classical counterpart*, and bake in regularisation (parameter clamping, weight decay in angle-space, or aggressive target-network updating) rather than hoping it is unnecessary.

### 3.2 Policy-gradient QRL

#### Jerbi, Gyurik, Marshall, Briegel & Dunjko 2021 — *Parametrized Quantum Policies for Reinforcement Learning*

- The canonical reference for quantum policies. Introduces two parametrisations:
  - **RAW-PQC**: the policy is the Born-rule distribution over computational basis states, partitioned into action subsets. Sampling from the policy *is* a quantum measurement.
  - **SOFTMAX-PQC**: per-action observables O_a are measured, and the policy is a softmax(β·⟨O_a⟩_{s,θ}) over their expectation values. This decouples expressivity from the output distribution and is the variant most subsequent work builds on.
- Adds two architectural features that have become standard practice:
  - **Trainable input scaling (λ)**, applied to every data-encoding gate. This enriches the Fourier spectrum of the model (cf. Schuld et al. 2021) without touching the encoding layout.
Replace $R(x)$ with $R(λ \cdot x)$ where λ is a trainable parameter. Now the encoding gate's eigenvalues effectively become $\pm λ/2$, so the accessible frequencies become multiples of $λ$. The optimizer learns the right $w$ for your data.
In practice people often go further and use $R(λ \cdot x + \theta)$ — trainable scale **and** bias, directly analogous to a classical neuron's $λx + b$.

<img src="https://i.imgur.com/GzbffN7.png" width="600">
  
  - **Trainable observables**, parametrised e.g. as weighted sums of computational-basis projections, with constraints to avoid the observables taking over training.
- Proves efficient sampling and gradient estimation for SOFTMAX-PQC policies and provides a formal quantum-advantage construction: there are tasks (based on discrete-log hardness) where SOFTMAX-PQC policies are learnable but no classical neural policy can learn them efficiently under standard cryptographic assumptions.
- Empirically solves CartPole, MountainCar, Acrobot with very few qubits.
- Takeaway for Q-IQL: if we quantise the *policy* π, the SOFTMAX-PQC parametrisation with trainable input scalings and trainable observables is the starting point.

#### Meyer et al. 2023 — *Quantum natural policy gradients*

- Extends the Jerbi et al. line to natural-gradient updates using the quantum Fisher information. Reports sample-efficiency improvements on small benchmarks. Relevant as a pointer rather than a likely direct ingredient for Q-IQL, since IQL extracts the policy by advantage-weighted BC rather than by policy gradient.

#### Sequeira et al. 2024 — *Trainability issues in quantum policy gradients*

- A companion to the Franz et al. stability paper, but for policy-gradient agents. Documents gradient-variance collapse in quantum policies and links it to the quantum-RL-specific barren-plateau regime.

### 3.3 Actor–critic QRL

#### Lan 2021 — *Variational Quantum Soft Actor-Critic*

- Proposes a VQC-based SAC for *continuous* action spaces. The policy is a hybrid quantum–classical network: a VQC feeds into a small classical MLP that produces the means and log-stds of a Gaussian policy.
- Benchmarked on standard Gym continuous-control tasks. The quantum variant matches classical SAC with substantially fewer tunable parameters, though at higher wall-clock cost because of circuit simulation.
- Notes the *entanglement of quantum contribution with classical head*: results depend strongly on the size of the classical post-processing net, so the paper cannot cleanly attribute performance gains to the quantum part. This is a recurring attribution problem in hybrid QRL.

#### Kölle et al. 2024 — *Quantum Advantage Actor-Critic for Reinforcement Learning*

- Splits A2C into four configurations — classical actor + quantum critic, quantum actor + classical critic, both quantum, both classical — on CartPole.
- Finding: hybrid variants (quantum actor OR quantum critic, but not both) with small classical post-processing significantly outperform both the fully classical and fully quantum baselines at matched parameter counts. Fully quantum lags due to NISQ limitations.
- Takeaway for Q-IQL: *if only one of {V, Q, π} is quantised, the classical post-processing on top of the quantum output is not optional* — it compensates for bounded-range readouts and noise, and the choice of which component to quantise matters more than whether to quantise.

#### Wu et al. 2020/2025 — *Quantum Reinforcement Learning in Continuous Action Space*

- A quantum DDPG tailored to the specific setting of *quantum control targets* (the environment state is a quantum state, the action is a unitary). Not a generic MuJoCo-style DDPG, but relevant as evidence that continuous-action QRL is tractable when the architecture is chosen carefully.

#### Other actor–critic-flavoured work

- Chen et al. 2022 — evolutionary optimisation of VQC policies (gradient-free alternative for small agents).
- Kölle et al. 2024 (QSW) — optimisation-technique ablations (Adam, SPSA, etc.) for VQC-RL.
- Acuto et al. 2022 — continuous-action VQC-RL with additional classical heads.

### 3.4 Offline QRL

#### Cheng, Zhang, Shen & Tao 2023 — *Offline Quantum Reinforcement Learning in a Conservative Manner*

**This is the most directly relevant paper in the literature for Q-IQL.** To our knowledge it is the only published offline QRL algorithm.

Key design choices:

- **Architecture.** Q(s, a) is represented by a VQC *augmented* with two features borrowed from the post-Jerbi PQC tradition:
  - *Data re-uploading* for expressivity.
  - A *trainable output scaling parameter* so the bounded ⟨Z⟩ readout can reach the (potentially large) Q-value range produced by offline Bellman backups. This is identical in spirit to Skolik et al. 2022's output scaling.

- **Method.** CQ2L is a quantum analogue of Conservative Q-Learning (CQL, Kumar et al. 2020), not of IQL:
  - A double-Q scheme (two VQC Q-networks, take the min on the target side) reduces overestimation bias.
  - A conservatism penalty explicitly pushes down Q-values on actions *not* in the offline dataset.
  
- **Empirical observation.** The authors show that naïve "offline VQ-DQN" (just running VQ-DQN on fixed offline data without conservatism) fails on tasks online VQ-DQN can solve — *the same distributional-shift pathology that motivates offline RL classically also afflicts its quantum counterpart*. Only with the conservatism penalty does offline training recover online performance.

- **Benchmarks.** Small discrete-action environments (e.g. CartPole with offline datasets generated by a separate policy). No D4RL-MuJoCo.

Relationship to Q-IQL:

- CQ2L confirms that the core problem IQL was designed to solve — bootstrapping from out-of-distribution actions — *is* a problem for quantum value functions too, and is not accidentally "solved" by the bounded range of ⟨Z⟩ observables.
- However, IQL and CQL are fundamentally different solutions to that problem. IQL never queries the Q-function on OOD actions in the first place; it does in-sample expectile regression on V(s). A quantum IQL cannot simply copy CQ2L's conservatism penalty — it has to implement the expectile loss on the V-network and AWR on π. That is precisely why Q-IQL is a new contribution, not a minor variant of CQ2L.

#### *Batch Quantum Reinforcement Learning*, Periyasamy et al. (2023)

- In this work, Periyasamy et al. propose batch-constrained quantum Q-learning (BCQQ), a offline QRL algorithm based on the classical discrete batch-constrained deep Q-learning
(BCQ) algorithm by Fujimoto et al. [Fuj+19]. Furthermore, the authors introduce a novel data re-uploading (DRU) scheme, which they call cyclic DRU. Experiments are executed in the OpenAI CartPole environment.

<img src="https://i.imgur.com/Uk5QD2R.png" width="1000">


- Result: Moreover, the cumulative reward these models can achieve beyond 500 is tested, which shows that the VQC with cyclic DRU out-performs the VQC with standard DRU


## 4. Component-level quantization patterns

Across the papers above, here is how each classical RL component has been replaced by a VQC:

| RL component | Has been quantised? | How | Representative work |
|---|---|---|---|
| Q(s, a) — discrete actions | Yes, extensively | VQC with one Pauli-Z readout per action; trainable output scaling | Chen 2020; Lockwood & Si 2020; Skolik 2022; Cheng 2023 |
| Q(s, a) — continuous actions | Rarely; usually via a critic with state-only input and action fed in separately | VQC taking (s, a) as input, single-observable readout | Lan 2021 (VQ-SAC critic) |
| V(s) | Rare as a standalone object | VQC with one scalar observable; or a critic head over state | Lan 2021; Kölle 2024 |
| π(a|s) — discrete | Yes, well-established | RAW-PQC (Born rule over action subsets) or SOFTMAX-PQC (softmax over per-action observables) | Jerbi 2021; Meyer 2023 |
| π(a|s) — continuous, Gaussian | Yes, hybrid only | VQC output → small classical head → (μ, log σ) of a Gaussian | Lan 2021; Acuto 2022 |
| A(s, a) = Q − V | **Not quantised as a standalone object in any paper we found.** Advantage is computed by taking the difference of a quantum Q and a quantum (or classical) V | — | — |

Two practical observations for Q-IQL:

- **The advantage has never been represented as a native quantum observable.** IQL computes A(s, a) = Q(s, a) − V(s). If V and Q are both VQCs there is no reason the advantage has to be computed classically — one could in principle read a joint observable proportional to (Q − V) — but there is no published work exploring this.
- **Advantage-weighted BC of a policy has never been implemented on top of a quantum policy.** Jerbi et al.'s SOFTMAX-PQC is trained by REINFORCE-style policy gradients; CQ2L is value-only and extracts actions by argmax. Q-IQL's third loss (AWR on π) would therefore be a novel combination: SOFTMAX-PQC-style policy, trained by weighted maximum likelihood against dataset actions, with weights coming from a VQC-derived advantage.

---

## 5. Design building blocks

### 5.1 State encoding

Two encoding families appear repeatedly:

- **Angle / Pauli-rotation encoding.** s_i is the angle of a Pauli rotation on qubit i. Shallow (depth 1) and fast. Only accesses frequency 1 in the model's Fourier decomposition, so on its own it is limited.
- **Data re-uploading (Pérez-Salinas et al. 2020).** Interleaves encoding layers with trainable layers and repeats L times. Schuld, Sweke & Meyer (2021) prove this produces a partial Fourier series in s whose frequency spectrum scales with L, enabling universal approximation in the limit. This is the encoding used by Skolik et al. 2022, Jerbi et al. 2021 (with trainable input scalings λ) and CQ2L.

For D4RL / MuJoCo state dimensions (11–17 features for Hopper/Walker2d), angle encoding on n ≈ 6–8 qubits with data re-uploading at L = 3–5 is the standard starting point.

### 5.2 Ansatz choice

- **Hardware-efficient ansätze** — single-qubit rotations interleaved with a ring of CNOTs or CZs. Cheap, shallow, but prone to barren plateaus as the number of qubits or layers grows (McClean et al. 2018).
- **Problem-inspired / equivariant ansätze** — incorporate known symmetries of the task (e.g. Skolik et al. 2023 for graph structure). Less noise-sensitive, more trainable, but require us to identify the right symmetry. For generic D4RL-locomotion tasks there is little obvious symmetry to exploit beyond coordinate permutation of matched joints.
- **Layerwise / growing ansätze** (Skolik, McClean et al. 2021) — train a shallow circuit first and progressively add layers. Mitigates barren plateaus. A reasonable robustness insurance for Q-IQL if trainability becomes a blocker.

### 5.3 Measurement & readout

- **Local observables beat global ones for trainability.** Cerezo et al. 2021 prove that with a global cost function gradients vanish exponentially in qubit count for shallow circuits, while with local observables gradients vanish at worst polynomially. All modern QRL papers therefore use local Pauli observables.
- **Trainable output scaling is essentially mandatory for value-based methods.** ⟨Z⟩ ∈ [−1, 1], but Q-values in an offline Bellman backup can be large (hundreds or thousands in D4RL). A scalar (or per-action) trainable output scaling parameter is used by Skolik et al. 2022, CQ2L and all subsequent value-based QRL work.
- **Trainable observables.** Jerbi et al. 2021 parametrise O_a itself and train it jointly; Skolik et al. 2023 study robustness of such observables under hardware errors. Useful but adds trainable parameters — care must be taken that the observable does not absorb all learning and reduce the PQC to a random feature extractor.

---

## 6. Current limitations

### 6.1 Barren plateaus and trainability

- **Random-circuit barren plateaus** (McClean et al. 2018): for sufficiently deep random PQCs forming approximate 2-designs, the variance of the cost gradient decays exponentially with qubit count. Random initialisation of a deep VQC on ≥ 10 qubits is almost certainly not trainable.
- **Cost-function-dependent barren plateaus** (Cerezo et al. 2021): global observables cause exponential gradient decay even at *shallow* depths; local observables are safe up to O(log n) depth.
- **Noise-induced barren plateaus** (Wang et al. 2021): depolarising noise flattens the loss landscape exponentially in circuit depth regardless of the cost function. Real-hardware deployment compounds the trainability problem.
- **Mitigations.** Layerwise training (Skolik, McClean et al. 2021), smart initialisations (Grant et al. 2019), entanglement control, small circuits, local observables. None of these are free — they constrain the architecture.

### 6.2 Shot and hardware noise

- Every quantum expectation value is estimated from a finite number of shots; the shot count enters Q-learning as a source of bootstrapping bias.
- Skolik, Mangini, Bäck, Macchiavello & Dunjko (2023) study robustness of quantum RL under hardware errors and find that shallow circuits with trainable scalings tolerate moderate noise reasonably well; deep circuits degrade sharply.

### 6.3 Q-learning-specific instabilities (VQ-DQN)

- Franz et al. 2023 (covered in §3.1) document unbounded parameter drift under periodic loss landscapes and non-reproducibility across seeds. Treat as a default-expected failure mode for VQ-DQN-style training, not a corner case.

### 6.4 Scalability

- Published QRL papers essentially all run at ≤ 8 qubits on ≤ 10-dimensional state spaces. The jump to D4RL locomotion (11–17 dimensions, up to 6-dimensional actions, sparse and high-magnitude rewards) is a real research step, not a drop-in scaling.
- Classical simulation of VQC training dominates wall-clock cost; batches of size 256 through an 8-qubit data-reuploading circuit are several orders of magnitude slower than an MLP at comparable parameter count on a GPU. This affects how large a replay batch and how many gradient steps are affordable.

### 6.5 Attribution

- In hybrid architectures where a classical MLP post-processes the VQC output (Lan 2021, Kölle 2024), it is routinely unclear whether the quantum component is contributing expressivity or merely acting as a fixed random feature map. Any Q-IQL paper should control for this by ablating the classical head.

---

## 7. Open problems and implications for Q-IQL

### 7.1 The gap the project sits in

There is:

- a mature body of VQ-DQN-style work for *online, discrete-action* tasks;
- few offline-QRL algorithm (CQ2L) that reproduces a *CQL*-style conservatism penalty with a VQC Q-function;
- policy-gradient VQCs (RAW-PQC / SOFTMAX-PQC) for *online* REINFORCE-style training;
- no published work that:
  - uses expectile regression for a quantum V(s),
  - uses advantage-weighted BC for a quantum policy,
  - combines a VQC V, a VQC Q, and a VQC policy in a single in-sample offline loop,
  - evaluates VQC-based offline RL on D4RL.

Q-IQL therefore occupies genuine white space. The closest neighbours are CQ2L (offline, VQC Q only, CQL-style) and Skolik et al. 2022 (online, VQC Q, careful observable / scaling design).

### 7.2 What CQ2L already tells:

- **Offline distributional shift is a real problem for quantum Q-networks.** Do not assume the bounded range of ⟨Z⟩ or the over-parametrisation of the ansatz will "regularise it away" — CQ2L's ablation of plain offline VQ-DQN shows it does not.
- **Data re-uploading + trainable output scaling is the minimum viable Q-function backbone.** Anything thinner than that is likely to underfit or to be unable to reach offline Q-value magnitudes.
- **Double-Q machinery is compatible with VQCs.** Two VQC Q-networks trained in parallel is not more expensive per step than one — it is exactly the classical double-Q overhead.

### 7.3 Which IQL component to quantise first

Based on the survey, the highest-expected-value single-component quantisations are, in decreasing order of prior support:

1. **Q(s, a) only.** Directly reusable architecture from Skolik et al. 2022 + CQ2L. The V-network stays an MLP (so expectile regression is a standard classical loss), the policy stays an MLP with classical AWR. Maximum compatibility with existing literature; minimum novelty.
2. **V(s) only.** Expectile regression on a VQC V(s) is a new construction, but the loss is well-defined and shot noise enters symmetrically. Q stays classical and the bootstrapping is classical. Good middle ground and arguably the component that would most benefit from a Fourier-basis representation (V(s) can be non-monotonic in s across the MDP, and the data re-uploading Fourier spectrum is a natural match).
3. **π(a|s) only.** Natural fit with SOFTMAX-PQC. AWR-weighted-MLE of a SOFTMAX-PQC policy is not published; this is the most novel of the three.
4. **Multiple components jointly.** The Kölle et al. 2024 finding is important here: on CartPole, fully-quantum A2C *underperforms* hybrid variants. We should almost certainly not make V, Q and π all quantum in the first iteration.

A plausible first architecture is therefore: **classical V(s) with expectile regression → classical/VQC Q(s, a) with IQL SARSA-style backup → classical or SOFTMAX-PQC π(a|s) trained by AWR.** Which of V, Q, π is the VQC then becomes the ablation axis of the Q-IQL paper.

### 7.4 Encoding, ansatz and readout defaults

Drawing on §5 and on CQ2L:

- **Encoding.** Angle encoding with trainable input scalings (Jerbi 2021) and L ≈ 3–5 data re-uploading layers. Normalise s to [−π, π] before encoding.
- **Ansatz.** Hardware-efficient ansatz with a linear CNOT chain. Start with 6–8 qubits for Hopper-style state dimensions. Use layerwise training if barren plateaus occur.
- **Readout.** Local Pauli-Z observables, one per output dimension (per-action for discrete Q; a single observable for V; per-action-component for continuous Q). Always include a trainable output scaling (and offset) on top.
- **Regularisation.** Because of the Franz et al. parameter-drift pathology, add angle-space weight decay or periodic wrapping to the optimizer from day one. Do not reproduce their failure mode.

### 7.5 Benchmarking

- **Baselines.** Classical IQL at matched parameter counts; CQ2L (offline VQC-CQL) if reimplemented; classical BC as a floor; random policy as an absolute floor.
- **Metrics.** Beyond final return, track parameter-drift magnitude and gradient variance across training — these are the things the literature says will go wrong and that papers like Franz et al. had to dig up after the fact.

### 7.6 Research questions worth calling out for the paper

- Does the Fourier-basis structure of a data-reuploading VQC make it a better fit for V(s) (value functions that are fundamentally functions of state, often smooth) than for Q(s, a) (which tends to have sharp edges at action-boundaries)?
- Does the boundedness of ⟨Z⟩ observables interact usefully with IQL's expectile regression? Expectile regression with τ close to 1 produces large target values for good actions; trainable output scaling has to compensate, and this compensation is a new learning dynamic that does not exist in classical IQL.
- Can advantage-weighted BC of a SOFTMAX-PQC policy be shown to enjoy the same information-theoretic separations Jerbi et al. prove for REINFORCE-trained SOFTMAX-PQC? This would be a genuinely new theoretical contribution.
- Does offline training *help* with barren plateaus (because trajectories are fixed and do not change distributionally over time) or *hurt* (because there is no exploration to escape a flat region)? CQ2L does not answer this question.

---

## 8. References

- Kostrikov, Nair & Levine. Offline Reinforcement Learning with Implicit Q-Learning. arXiv:2110.06169 (2021). ICLR 2022.
- Kumar, Zhou, Tucker & Levine. Conservative Q-Learning for Offline Reinforcement Learning. arXiv:2006.04779 (2020). NeurIPS 2020.
- Fu et al. D4RL: Datasets for Deep Data-Driven Reinforcement Learning. arXiv:2004.07219 (2020).
- Meyer, Ufrecht, Periyasamy, Scherer, Plinge & Mutschler. A Survey on Quantum Reinforcement Learning. arXiv:2211.03464 (2022, rev. 2024).
- Jerbi, Gyurik, Marshall, Briegel & Dunjko. Parametrized Quantum Policies for Reinforcement Learning. NeurIPS 2021. arXiv:2103.05577.
- Skolik, Jerbi & Dunjko. Quantum Agents in the Gym: a variational quantum algorithm for deep Q-learning. Quantum 6, 720 (2022). arXiv:2103.15084.
- Chen, Yang, Qi, Chen, Ma & Goan. Variational Quantum Circuits for Deep Reinforcement Learning. IEEE Access 8, 141007–141024 (2020). arXiv:1907.00397.
- Lockwood & Si. Reinforcement Learning with Quantum Variational Circuit. AAAI AIIDE 16, 245–251 (2020). arXiv:2008.07524.
- Cheng, Zhang, Shen & Tao. Offline Quantum Reinforcement Learning in a Conservative Manner. AAAI 37(6), 7148–7156 (2023). [CQ2L]
- Lan. Variational Quantum Soft Actor-Critic. arXiv:2112.11921 (2021).
- Kölle, Feist, Stein, Wölckert & Linnhoff-Popien. Quantum Advantage Actor-Critic for Reinforcement Learning. arXiv:2401.07043 (2024).
- Wu, Jin, Wen, Han & Wang. Quantum Reinforcement Learning in Continuous Action Space. Quantum 9, 1660 (2025). arXiv:2012.10711.
- Franz, Wolf, Periyasamy, Ufrecht, Scherer, Plinge, Mutschler & Mauerer. Uncovering Instabilities in Variational-Quantum Deep Q-Networks. Journal of the Franklin Institute 360(17), 13822 (2023). arXiv:2202.05195.
- Schuld, Sweke & Meyer. Effect of data encoding on the expressive power of variational quantum machine learning models. Phys. Rev. A 103, 032430 (2021). arXiv:2008.08605.
- Pérez-Salinas, Cervera-Lierta, Gil-Fuster & Latorre. Data re-uploading for a universal quantum classifier. Quantum 4, 226 (2020). arXiv:1907.02085.
- McClean, Boixo, Smelyanskiy, Babbush & Neven. Barren plateaus in quantum neural network training landscapes. Nature Communications 9 (2018).
- Cerezo, Sone, Volkoff, Cincio & Coles. Cost function dependent barren plateaus in shallow parametrized quantum circuits. Nature Communications 12, 1791 (2021).
- Wang, Fontana, Cerezo, Sharma, Sone, Cincio & Coles. Noise-induced barren plateaus in variational quantum algorithms. Nature Communications 12 (2021).
- Skolik, McClean, Mohseni, van der Smagt & Leib. Layerwise learning for quantum neural networks. Quantum Machine Intelligence 3, 5 (2021).
- Skolik, Mangini, Bäck, Macchiavello & Dunjko. Robustness of quantum reinforcement learning under hardware errors. EPJ Quantum Technology 10, 8 (2023).
- Meyer, Scherer, Plinge, Mutschler & Hartmann. Quantum Natural Policy Gradients. IEEE QCE (2023).
- Sequeira, Santos & Barbosa. Trainability issues in quantum policy gradients. Machine Learning: Science and Technology 5, 035037 (2024).
- Dunjko, Taylor & Briegel. Advances in quantum reinforcement learning. IEEE SMC 2017.
- Wang, Sundaram, Kothari, Kapoor & Roetteler. Quantum Algorithms for Reinforcement Learning with a Generative Model. ICML 2021.
- Cherrat, Kerenidis & Prakash. Quantum reinforcement learning via policy iteration. Quantum Machine Intelligence 5, 30 (2023).

---